# AegisPlan: Fast QLoRA Fine-Tuning with Unsloth
**Autonomous Neuro-Symbolic Healthcare Spatial Synthesis Engine**

This notebook fine-tunes an LLM to generate code-compliant `FML-DSL` clinical floorplans using **Unsloth** (2x faster, 70% less VRAM).
Runs on **Google Colab Free Tier (T4 16GB GPU)** or **Colab Pro (L4 / A100)**.

### Step 1: Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth "xformers<0.0.29" trl peft accelerate bitsandbytes datasets

### Step 2: Load Model with Unsloth 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto detection (Float16 on T4, Bfloat16 on Ampere+)
load_in_4bit = True

# Select base model:
# 1. "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" (Fastest & best structured code generation on T4)
# 2. "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
# 3. "nvidia/NVIDIA-Nemotron-Nano-9B-v2" (Requires A100 / L4 on Colab Pro)
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Configure LoRA (r=32, alpha=64 from Master Plan Section 5)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

### Step 3: Load Certified Train / Validation / Test Splits
Upload `train.jsonl`, `val.jsonl`, and `test.jsonl` (from `dataset/`) using the Colab file browser (left panel).

In [ ]:
from datasets import load_dataset
import os

if os.path.exists("train.jsonl") and os.path.exists("val.jsonl"):
    train_dataset = load_dataset("json", data_files="train.jsonl", split="train")
    val_dataset = load_dataset("json", data_files="val.jsonl", split="train")
    print(f"Loaded verified train split: {len(train_dataset)} samples")
    print(f"Loaded verified val split:   {len(val_dataset)} samples")
elif os.path.exists("sft_clinical_curriculum.jsonl"):
    full_ds = load_dataset("json", data_files="sft_clinical_curriculum.jsonl", split="train")
    splits = full_ds.train_test_split(test_size=0.10, seed=42)
    train_dataset = splits["train"]
    val_dataset = splits["test"]
    print(f"Auto-split curriculum into {len(train_dataset)} train, {len(val_dataset)} val.")
else:
    raise FileNotFoundError("Please upload train.jsonl & val.jsonl via the Colab files panel!")

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_formatted = train_dataset.map(formatting_prompts_func, batched=True)
val_formatted = val_dataset.map(formatting_prompts_func, batched=True)
print("Train & Val formatted successfully!")

### Step 4: Run Fine-Tuning with SFTTrainer & Validation Evaluation

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="aegisplan_checkpoints",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("Training Complete!")

### Step 5: Test Generation on Held-Out Test Set

In [ ]:
FastLanguageModel.for_inference(model)
import json

if os.path.exists("test.jsonl"):
    with open("test.jsonl", "r") as f:
        test_sample = json.loads(f.readline())
    test_messages = test_sample["messages"][:2]
    print("Using held-out test brief:", test_messages[1]["content"][:120], "...")
else:
    test_messages = [
        {"role": "system", "content": "You are AegisPlan, an autonomous AI architect specializing in U.S. Healthcare Outpatient Facilities.\nOutput ONLY valid FML-DSL code matching the exact grammar: BOUNDARY RECT 0 0 <W> <H>, ROOM, DOOR, FDOOR, END_PLAN."},
        {"role": "user", "content": "Generate an outpatient clinic floorplan for boundary RECT 0 0 16000 12000.\nClinical Brief: Design a 16m x 12m Primary care outpatient clinic requiring central circulation corridor, 3 outpatient exam rooms (>= 11.15 m2 each), single-user ADA compliant restroom, clean and soiled utility support rooms.\nEmit strictly valid FML-DSL:"}
    ]

inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)
generated_fml = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n=== GENERATED FML-DSL OUTPUT ===")
print(generated_fml)

### Step 6: Save Model & Export (LoRA / Hugging Face Hub / GGUF)

In [ ]:
# 1. Save LoRA Adapter Locally
model.save_pretrained("aegisplan_lora_adapter")
tokenizer.save_pretrained("aegisplan_lora_adapter")
print("Saved LoRA adapter to ./aegisplan_lora_adapter")

# 2. Optional: Push to Hugging Face Hub
# from huggingface_hub import login
# login()
# model.push_to_hub_merged("your-username/aegisplan-clinical-7b", tokenizer, save_method="merged_16bit")

# 3. Optional: Export directly to GGUF for llama.cpp / vLLM
# model.save_pretrained_gguf("aegisplan_gguf", tokenizer, quantization_method="q4_k_m")